# Comprehensive OCR Architecture Experiments

This notebook benchmarks **four custom neural network architectures** for OCR from scratch to find the lowest possible Character Error Rate (CER).

The contenders are
1. Standard CRNN with Bidirectional LSTMs
2. Fully Convolutional Network using TCNs
3. Sequence to Sequence with Self Attention
4. Pure Vision Transformer Encoder

All models share the same CNN front-end, CTC loss layer, and evaluation callback, making the comparison fair and direct.

In [ ]:
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from jiwer import cer

from google.colab import drive  # type: ignore
import os
import zipfile

In [ ]:
drive.mount('/content/drive')

ZIP_PATH     = '/content/drive/MyDrive/cig_ps.zip'
EXTRACT_PATH = '/content/cig_ps_local'

if not os.path.exists(EXTRACT_PATH):
    print(f"Extracting {ZIP_PATH} to local storage...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Extraction complete.")
else:
    print("Data already present locally — skipping extraction.")

DATA_DIR          = Path(EXTRACT_PATH)
TRAIN_IMAGES_DIR  = DATA_DIR / "cig_ps/train_images"
TRAIN_CSV_PATH    = DATA_DIR / "cig_ps/train-labels.csv"
TEST_IMAGES_DIR   = DATA_DIR / "cig_ps/test_images"

print(f"Train images : {TRAIN_IMAGES_DIR}")
print(f"Train CSV    : {TRAIN_CSV_PATH}")

In [ ]:
# PAD_TOKEN must be:
#   (a) larger than the vocabulary size so it never collides with a real char index
#   (b) a positive integer because keras.backend.ctc_batch_cost expects positive labels
PAD_TOKEN = 99

BATCH_SIZE = 64

# Image dimensions — all images are resized to this shape before entering the network
IMG_WIDTH  = 200
IMG_HEIGHT = 100

In [ ]:
df = pd.read_csv(TRAIN_CSV_PATH)
df['image_path'] = df['image'].apply(lambda x: str(TRAIN_IMAGES_DIR / x))
df['label'] = df['text']

print(f"Total samples : {len(df)}")
df.head()

In [ ]:
characters = sorted(set(char for label in df['text'].astype(str) for char in label))

print(f"Vocabulary size : {len(characters)}")
print(f"Characters      : {''.join(characters)}")

In [ ]:
char_to_num = layers.StringLookup(
    vocabulary = list(characters),
    mask_token = None
)

num_to_char = layers.StringLookup(
    vocabulary = char_to_num.get_vocabulary(),
    mask_token = None,
    invert = True
)

In [ ]:
def encode_single_sample(img_path, label):
    """
    Reads one image + label pair and returns a dict ready for batching.

    Image pipeline:
      1. Read raw bytes from disk.
      2. Decode to grayscale float32 in [0, 1].
      3. Resize to (IMG_HEIGHT, IMG_WIDTH).
      4. Transpose to (IMG_WIDTH, IMG_HEIGHT, 1) so the width axis aligns
         with the temporal/sequence axis expected by the RNN/Conv1D layers.

    Label pipeline:
      1. Split the UTF-8 string into individual characters.
      2. Map each character to its integer index via char_to_num.
      3. Cast to float32 — required by keras.backend.ctc_batch_cost.
    """
    img = tf.io.read_file(img_path)
    img = tf.io.decode_png(img, channels = 1)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = tf.transpose(img, perm=[1, 0, 2])
    label = char_to_num(tf.strings.unicode_split(label, input_encoding="UTF-8"))
    label = tf.cast(label, tf.float32)
    return {"image": img, "label": label}

In [ ]:
SPLIT_IDX = 17600

train_df = df.iloc[:SPLIT_IDX]
val_df   = df.iloc[SPLIT_IDX:]

train_images = train_df['image_path'].values
train_labels = train_df['text'].astype(str).values

val_images   = val_df['image_path'].values
val_labels   = val_df['text'].astype(str).values

print(f"Training samples   : {len(train_images)}")
print(f"Validation samples : {len(val_images)}")

train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))  # type: ignore
train_dataset = (
    train_dataset
    .map(encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .padded_batch(
        BATCH_SIZE,
        padding_values={"image": 0.0, "label": tf.cast(PAD_TOKEN, tf.float32)}
    )
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

validation_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_labels))  # type: ignore
validation_dataset = (
    validation_dataset
    .map(encode_single_sample, num_parallel_calls=tf.data.AUTOTUNE)
    .padded_batch(
        BATCH_SIZE,
        padding_values={"image": 0.0, "label": tf.cast(PAD_TOKEN, tf.float32)}
    )
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

In [ ]:
class CTCLayer(layers.Layer):
    """
    Custom Keras layer that computes CTC loss during the forward pass.

    Keras does not expose CTC as a standard loss function, so we embed it
    inside a layer via add_loss().  The layer is transparent at inference
    time — it simply returns y_pred unchanged.

    Label lengths are computed dynamically by counting non-PAD_TOKEN entries,
    which means we never need to pass explicit length tensors to the model.
    """

    def __init__(self, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.loss_fn = keras.backend.ctc_batch_cost

    def call(self, y_true, y_pred):  # type: ignore
        batch_len    = tf.cast(tf.shape(y_true)[0], dtype="int64")
        input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")

        label_length = tf.reduce_sum(
            tf.cast(tf.not_equal(y_true, PAD_TOKEN), tf.int64), axis=-1
        )

        label_length = tf.reshape(label_length, (batch_len, 1))
        input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
        loss = self.loss_fn(y_true, y_pred, input_length, label_length)
        self.add_loss(loss)

        return y_pred

In [ ]:
class CERMetricCallback(keras.callbacks.Callback):
    """
    Computes Character Error Rate (CER) on a sample of the validation set
    at the end of every epoch and stores it in the Keras history object
    so it can be plotted alongside the training loss.

    CTC decoding:
      - Uses greedy (best-path) decoding for speed.
      - keras.backend.ctc_decode pads output sequences with -1 (not PAD_TOKEN),
        so we filter on -1 when extracting predicted characters.
    """

    def __init__(self, dataset, num_to_char, prediction_model):
        super().__init__()
        self.dataset          = dataset
        self.num_to_char      = num_to_char
        self.prediction_model = prediction_model

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}

        true_texts = []
        pred_texts = []

        for batch in self.dataset.take(5):
            images = batch["image"]
            labels = batch["label"]

            preds = self.prediction_model.predict(images, verbose=0)

            input_len   = np.ones(preds.shape[0]) * preds.shape[1]
            decoded, _  = keras.backend.ctc_decode(preds, input_length=input_len, greedy=True)
            decoded_seq = decoded[0]

            for i in range(len(images)):
                true_indices = labels[i][labels[i] != PAD_TOKEN]
                true_text    = (
                    tf.strings.reduce_join(self.num_to_char(true_indices))
                    .numpy().decode("utf-8")
                )

                true_texts.append(true_text)
                pred_indices = decoded_seq[i][decoded_seq[i] != -1]
                pred_text    = (
                    tf.strings.reduce_join(self.num_to_char(pred_indices))
                    .numpy().decode("utf-8")
                )
                pred_texts.append(pred_text)

        epoch_cer = cer(true_texts, pred_texts)
        logs["val_cer"] = epoch_cer # type: ignore
        print(f"\nEpoch {epoch + 1} — Validation CER: {epoch_cer:.4f}")

In [ ]:
# ── 9a. Shared CNN Feature Extractor ─────────────────────────────────────────
# This block is duplicated inside each builder to keep the builders self-contained.
# Two Conv-Pool stages reduce spatial dims by 4× in each direction and produce
# 64 feature maps, then a Dense projection prepares the sequence for the RNN/Attention head.

def _cnn_backbone(input_img):
    """Returns the reshaped feature sequence from the shared CNN front-end."""
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(input_img)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)

    new_shape = (IMG_WIDTH // 4, (IMG_HEIGHT // 4) * 64)
    x = layers.Reshape(target_shape=new_shape)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    return x


# Architecture 1: Baseline CRNN with Bidirectional LSTMs

def build_baseline_crnn(vocab_size):
    """
    Classic CRNN: CNN backbone → two stacked Bidirectional LSTMs → CTC.
    This is the well-established baseline for sequence OCR tasks.
    """
    input_img = layers.Input(shape=(IMG_WIDTH, IMG_HEIGHT, 1), name="image",  dtype="float32")
    labels    = layers.Input(shape=(None,),                    name="label",  dtype="float32")

    x = _cnn_backbone(input_img)

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25))(x)
    x = layers.Bidirectional(layers.LSTM(64,  return_sequences=True, dropout=0.25))(x)

    x = layers.Dense(vocab_size + 1, activation="softmax", name="dense_out")(x)
    output = CTCLayer(name="ctc_loss")(labels, x) # type: ignore

    model = keras.models.Model(
        inputs=[input_img, labels], outputs=output, name="Baseline_CRNN"
    )
    model.compile(optimizer=keras.optimizers.Adam(), jit_compile=False)
    return model


# Architecture 2: Fast CNN with Temporal Convolutions (TCN-style)

def build_tcn_model(vocab_size):
    """
    Replaces recurrent layers with two Conv1D + BatchNorm blocks.
    Fully convolutional → much faster training, but lacks explicit
    sequential memory across long sequences.
    """
    input_img = layers.Input(shape=(IMG_WIDTH, IMG_HEIGHT, 1), name="image", dtype="float32")
    labels    = layers.Input(shape=(None,),                    name="label", dtype="float32")

    x = _cnn_backbone(input_img)

    # Temporal convolutions — kernel_size=5 captures local context of 5 time steps
    x = layers.Conv1D(filters=128, kernel_size=5, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(filters=128, kernel_size=5, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.25)(x)

    x      = layers.Dense(vocab_size + 1, activation="softmax", name="dense_out")(x)
    output = CTCLayer(name="ctc_loss")(labels, x)

    model = keras.models.Model(
        inputs=[input_img, labels], outputs=output, name="TCN_Model"
    )
    model.compile(optimizer=keras.optimizers.Adam(), jit_compile=False)
    return model


# Architecture 3: CRNN + Multi-Head Self-Attention

def build_attention_model(vocab_size):
    """
    Extends the baseline CRNN with a Multi-Head Self-Attention layer placed
    after the first BiLSTM.  The residual connection (Add + LayerNorm) follows
    the standard Transformer encoder pattern.
    """
    input_img = layers.Input(shape=(IMG_WIDTH, IMG_HEIGHT, 1), name="image", dtype="float32")
    labels    = layers.Input(shape=(None,), name="label", dtype="float32")

    x = _cnn_backbone(input_img)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25))(x)
    attention = layers.MultiHeadAttention(num_heads=4, key_dim=128, dropout=0.2)(x, x)
    x = layers.Add()([x, attention])
    x = layers.LayerNormalization()(x)

    x      = layers.Dense(vocab_size + 1, activation="softmax", name="dense_out")(x)
    output = CTCLayer(name="ctc_loss")(labels, x)

    model = keras.models.Model(
        inputs=[input_img, labels], outputs=output, name="Attention_Model"
    )
    model.compile(optimizer=keras.optimizers.Adam(), jit_compile=False)
    return model


# Helper: Positional Embedding for the Transformer

class PositionalEmbedding(layers.Layer):
    """
    Adds learnable positional embeddings to a sequence of feature vectors.

    Without this, the Transformer has no sense of token order because
    self-attention is permutation-invariant by design.
    """

    def __init__(self, max_length, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.max_length = max_length
        self.embed_dim  = embed_dim
        self.pos_emb    = layers.Embedding(input_dim=max_length, output_dim=embed_dim)

    def call(self, inputs):
        positions = tf.range(start=0, limit=tf.shape(inputs)[1], delta=1)
        return inputs + self.pos_emb(positions)

    def get_config(self):
        config = super().get_config()
        config.update({"max_length": self.max_length, "embed_dim": self.embed_dim})
        return config


# Architecture 4: Pure Vision Transformer Encoder

def build_transformer_model(vocab_size):
    """
    Treats the CNN feature maps as a sequence of patches and processes them
    with a single Transformer encoder block (self-attention + FFN + residuals).
    No recurrence at all — relies entirely on attention for sequence modelling.
    """
    input_img = layers.Input(shape=(IMG_WIDTH, IMG_HEIGHT, 1), name="image", dtype="float32")
    labels    = layers.Input(shape=(None,),                    name="label", dtype="float32")

    x = layers.Conv2D(32, 3, activation="relu", padding="same")(input_img)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)

    seq_length = IMG_WIDTH  // 4
    embed_dim  = (IMG_HEIGHT // 4) * 64

    x = layers.Reshape(target_shape=(seq_length, embed_dim))(x)
    x = layers.Dense(128)(x)
    x = PositionalEmbedding(max_length=seq_length, embed_dim=128)(x)
    x = layers.Dropout(0.1)(x)

    attention = layers.MultiHeadAttention(num_heads=4, key_dim=128, dropout=0.1)(x, x)
    x = layers.Add()([x, attention])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(256, activation="relu")(x)
    ffn = layers.Dropout(0.1)(ffn)
    ffn = layers.Dense(128)(ffn)
    x   = layers.Add()([x, ffn])
    x   = layers.LayerNormalization()(x)

    x      = layers.Dense(vocab_size + 1, activation="softmax", name="dense_out")(x)
    output = CTCLayer(name="ctc_loss")(labels, x) # type: ignore

    model = keras.models.Model(
        inputs=[input_img, labels], outputs=output, name="Transformer_Model"
    )
    model.compile(optimizer=keras.optimizers.Adam(), jit_compile=False)
    return model

In [ ]:
vocab_size = char_to_num.vocabulary_size()
EPOCHS = 50

models_to_test = {
    "CRNN Baseline"      : build_baseline_crnn(vocab_size),
    "TCN Fast CNN"        : build_tcn_model(vocab_size),
    "Attention CRNN"      : build_attention_model(vocab_size),
    "Vision Transformer"  : build_transformer_model(vocab_size),
}

training_histories = {}

for model_name, model in models_to_test.items():
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"{'='*60}")

    pred_model = keras.models.Model(
        inputs=model.inputs[0],
        outputs=model.get_layer(name="dense_out").output
    )

    cer_callback = CERMetricCallback(
        dataset=validation_dataset,
        num_to_char=num_to_char,
        prediction_model=pred_model
    )

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=4,
        min_lr=1e-6
    )

    history = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=EPOCHS,
        callbacks=[early_stopping, cer_callback, reduce_lr],
        verbose=1
    )

    training_histories[model_name] = history

In [ ]:
plt.style.use("ggplot")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Architecture Comparison — Validation Metrics", fontsize=14, fontweight="bold")

for name, hist in training_histories.items():
    if "val_cer" in hist.history:
        axes[0].plot(hist.history["val_cer"], marker="o", label=name)

axes[0].set_title("Validation Character Error Rate")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("CER  (lower is better)")
axes[0].set_ylim(0, 1.0)
axes[0].legend()

for name, hist in training_histories.items():
    if "val_loss" in hist.history:
        axes[1].plot(hist.history["val_loss"], linestyle="--", label=name)

axes[1].set_title("Validation CTC Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("CTC Loss  (lower is better)")
axes[1].legend()

plt.tight_layout()
plt.show()